In [ ]:
# Se importan las librerías usadas durante el análisis.

from pathlib import Path

import pandas as pd
import plotly.express as px

In [ ]:
# Se definen las rutas de datos y entregables permanentes del taller.

data_file = Path("../data/salarios.csv")
submission_directory = Path("../submission")
submission_directory.mkdir(exist_ok=True)

In [ ]:
# Se carga el archivo y se verifican sus tipos, tamaño y campos disponibles.

salaries = pd.read_csv(data_file)
salaries.info()
salaries.shape
salaries.columns.tolist()

In [ ]:
# Se comprueba que cada registro representa un profesional y que los salarios son utilizables.

assert salaries["employee_id"].is_unique
assert salaries["salario_mensual_cop"].gt(0).all()
assert (
    salaries[["gerencia", "departamento", "trayectoria", "categoria"]]
    .notna()
    .all()
    .all()
)
salaries.groupby(["gerencia", "departamento"]).size()

In [ ]:
# Se agrupa la experiencia técnica en bandas comparables para definir pares internos.

salaries["banda_experiencia"] = pd.cut(
    salaries["experiencia_tecnica_anios"],
    bins=[-1, 4, 9, 14, 100],
    labels=["0-4", "5-9", "10-14", "15+"],
)
salaries.groupby("banda_experiencia", observed=True).size()

In [ ]:
# Se observa la distribución salarial por trayectoria; la mediana y el rango importan más que un único promedio.

salary_distribution = (
    salaries.groupby("trayectoria")
    .agg(
        profesionales=("employee_id", "size"),
        salario_mediano_cop=("salario_mensual_cop", "median"),
        percentil_10_cop=("salario_mensual_cop", lambda value: value.quantile(0.10)),
        percentil_90_cop=("salario_mensual_cop", lambda value: value.quantile(0.90)),
    )
    .reset_index()
)
salary_distribution.round(0)

In [ ]:
# Se comparan las distribuciones salariales de las trayectorias técnica y directiva.

fig = px.box(
    salaries,
    x="trayectoria",
    y="salario_mensual_cop",
    color="trayectoria",
    points=False,
    labels={
        "trayectoria": "Trayectoria",
        "salario_mensual_cop": "Salario mensual (COP)",
    },
    title="Distribución salarial por trayectoria",
    color_discrete_map={"Técnica": "#4e79a7", "Directiva": "#f28e2b"},
)
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

In [ ]:
# Se calcula la mediana empresarial de cada grupo de pares y la brecha individual frente a esa referencia.

peer_columns = ["categoria", "trayectoria", "banda_experiencia"]
salaries["salario_mediano_pares_cop"] = salaries.groupby(peer_columns, observed=True)[
    "salario_mensual_cop"
].transform("median")
salaries["brecha_vs_pares_pct"] = (
    salaries["salario_mensual_cop"] / salaries["salario_mediano_pares_cop"] - 1
)
salaries[
    peer_columns
    + ["salario_mensual_cop", "salario_mediano_pares_cop", "brecha_vs_pares_pct"]
].head()

In [ ]:
# ¿En qué áreas debemos revisar los salarios porque nuestros profesionales podrían ganar menos que perfiles similares en otras áreas de la empresa?

minimum_group_size = 30
department_gap = (
    salaries.groupby(["gerencia", "departamento"], as_index=False)
    .agg(
        profesionales=("employee_id", "size"),
        salario_mediano_cop=("salario_mensual_cop", "median"),
        brecha_mediana_vs_pares_pct=("brecha_vs_pares_pct", "median"),
        proporcion_mas_de_5_pct_debajo=(
            "brecha_vs_pares_pct",
            lambda value: (value <= -0.05).mean(),
        ),
    )
    .query("profesionales >= @minimum_group_size")
    .sort_values("brecha_mediana_vs_pares_pct")
)
department_gap.round(3)

In [ ]:
# Se visualiza la brecha mediana por departamento frente a perfiles comparables de toda la empresa.

department_gap["estado"] = department_gap["brecha_mediana_vs_pares_pct"].map(
    lambda value: "Por debajo de pares" if value < 0 else "En o sobre pares"
)
fig = px.bar(
    department_gap,
    x="brecha_mediana_vs_pares_pct",
    y="departamento",
    orientation="h",
    color="estado",
    text="profesionales",
    labels={
        "brecha_mediana_vs_pares_pct": "Brecha frente a pares",
        "departamento": "Departamento",
        "profesionales": "Profesionales",
    },
    title="Brecha salarial interna por departamento",
    color_discrete_map={
        "Por debajo de pares": "#e15759",
        "En o sobre pares": "#4e79a7",
    },
)
fig.add_vline(x=0, line_color="#333333")
fig.update_xaxes(tickformat=".0%")
fig.update_layout(template="plotly_white", legend_title_text="Estado")
fig.show()

In [ ]:
# Se priorizan áreas con una brecha mediana de al menos 5 % y una mayoría de profesionales bajo esa referencia.

areas_a_revisar = department_gap.query(
    "brecha_mediana_vs_pares_pct <= -0.05 and proporcion_mas_de_5_pct_debajo >= 0.50"
).copy()
areas_a_revisar[
    [
        "gerencia",
        "departamento",
        "profesionales",
        "brecha_mediana_vs_pares_pct",
        "proporcion_mas_de_5_pct_debajo",
    ]
].round(3)

In [ ]:
# Se proponen alternativas para revisar bandas, criterios de progresión y retención en las áreas priorizadas.

recommendations = areas_a_revisar.assign(
    alternativa_1="Revisar la banda salarial frente a perfiles comparables.",
    alternativa_2="Verificar criterios de progresión y ajustes salariales.",
    alternativa_3="Priorizar conversación de retención sin divulgar salarios individuales.",
)[
    [
        "gerencia",
        "departamento",
        "brecha_mediana_vs_pares_pct",
        "alternativa_1",
        "alternativa_2",
        "alternativa_3",
    ]
]
recommendations

In [ ]:
# Se guardan los resultados agregados para revisión posterior sin exponer información individual.

department_gap.drop(columns="estado").to_csv(
    submission_directory / "department_pay_gap.csv", index=False
)
recommendations.to_csv(
    submission_directory / "compensation_recommendations.csv", index=False
)

In [ ]:
# ¿Un profesional técnico puede alcanzar los salarios más altos sin tener que convertirse en directivo?

high_salary_threshold = salaries["salario_mensual_cop"].quantile(0.90)
salaries["salario_alto"] = salaries["salario_mensual_cop"].ge(high_salary_threshold)
career_path_summary = salaries.groupby("trayectoria", as_index=False).agg(
    profesionales=("employee_id", "size"),
    salario_mediano_cop=("salario_mensual_cop", "median"),
    profesionales_con_salario_alto=("salario_alto", "sum"),
    proporcion_con_salario_alto=("salario_alto", "mean"),
)
career_path_summary.round(3)

In [ ]:
# Se compara la proporción de profesionales en el decil salarial superior por trayectoria.

fig = px.bar(
    career_path_summary,
    x="trayectoria",
    y="proporcion_con_salario_alto",
    text="profesionales_con_salario_alto",
    labels={
        "trayectoria": "Trayectoria",
        "proporcion_con_salario_alto": "Proporción en salarios altos",
        "profesionales_con_salario_alto": "Profesionales",
    },
    title="Acceso a salarios altos por trayectoria",
    color="trayectoria",
    color_discrete_map={"Técnica": "#4e79a7", "Directiva": "#f28e2b"},
)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

In [ ]:
# Se identifica dónde se concentran los profesionales técnicos que ya alcanzan el decil salarial superior.

technical_high_salary = (
    salaries.query("trayectoria == 'Técnica' and salario_alto")
    .groupby(["categoria", "banda_experiencia"], observed=True, as_index=False)
    .agg(profesionales=("employee_id", "size"))
    .sort_values("profesionales", ascending=False)
)
technical_high_salary

In [ ]:
# Se documenta la respuesta gerencial y sus límites: las diferencias observadas no prueban por sí solas una causa.

conclusions = pd.DataFrame(
    {
        "pregunta": [
            "¿Existen áreas con posible brecha salarial interna?",
            "¿La dirección es la única vía hacia salarios altos?",
        ],
        "respuesta": [
            f"Sí. Se priorizan {', '.join(areas_a_revisar['departamento'])} para revisar bandas y criterios de progresión.",
            f"No. {int((salaries['trayectoria'].eq('Técnica') & salaries['salario_alto']).sum())} profesionales técnicos están en el decil salarial superior.",
        ],
        "límite": [
            "La brecha indica una señal para revisión; no identifica su causa por sí sola.",
            "La relación observada no prueba que una trayectoria cause el salario.",
        ],
    }
)
conclusions

In [ ]:
# Se guardan las conclusiones y la tabla de trayectorias para una discusión gerencial posterior.

career_path_summary.to_csv(
    submission_directory / "high_salary_by_career.csv", index=False
)
conclusions.to_csv(submission_directory / "analysis_conclusions.csv", index=False)